# CAD Wireframe → SolidWorks-Style Render  ·  v4 (Lean)

No vision model. Just ControlNet + SD with a universal prompt tuned for clean engineering renders.

**Storage:** ~6 GB on Drive (vs ~20 GB in v3)  
**Runtime:** ~3 min on T4  
**Only cell to edit:** Cell 3 (image path)

## Cell 1 — Install

In [ ]:
import subprocess, sys

# Install without touching the existing torch/torchvision stack
pkgs = [
    "diffusers>=0.27.0",
    "transformers>=4.40.0",
    "accelerate>=0.29.0",
    "controlnet-aux>=0.0.7",
    "huggingface_hub",
]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)

print("Done. Restart runtime (Runtime → Restart session), then run from Cell 2.")

## Cell 2 — Mount Drive & Setup Cache

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

CACHE_ROOT = Path("/content/drive/MyDrive/colab_model_cache")
CACHE_ROOT.mkdir(exist_ok=True)

PATHS = {
    "controlnet" : CACHE_ROOT / "controlnet-lineart",
    "sd15"       : CACHE_ROOT / "stable-diffusion-v1-5",
    "annotators" : CACHE_ROOT / "annotators",
}

print("Drive mounted. Cache:", CACHE_ROOT)
for name, path in PATHS.items():
    marker = path / ".download_complete"
    status = "cached" if marker.exists() else "will download on first run"
    print(f"  {name:15s}: {status}")

## Cell 3 — Config  ⬅ Only cell you need to edit

In [ ]:
# ══════════════════════════════════════════════════════════════════
INPUT_IMAGE_PATH = "/content/cad_wireframe.png"   # upload your file first
NUM_IMAGES       = 4
SEED             = 42   # change for different variations
# ══════════════════════════════════════════════════════════════════

# Universal prompt — works for any mechanical/industrial CAD drawing
# Tuned for: matte grey, white bg, sharp edges, SolidWorks aesthetic
PROMPT = (
    "SolidWorks 3D render of industrial mechanical equipment, "
    "smooth matte light grey surface, uniform soft studio lighting, "
    "pure white background, no shadows, no reflections, "
    "crisp sharp edges, clean geometry, perfect anti-aliasing, "
    "isolated product visualization, engineering render, "
    "CAD software screenshot, high detail, 4k"
)

NEGATIVE_PROMPT = (
    "blurry edges, soft edges, bloom, glow, chromatic aberration, "
    "metallic, shiny, chrome, mirror, reflections, specular highlights, "
    "photorealistic, real photo, background, environment, factory, floor, "
    "dark, dramatic lighting, shadows, rust, dirt, weathered, "
    "sketch, line art, wireframe, cartoon, painting, "
    "watermark, text, logo, low quality, deformed, blurry"
)

# Generation params
TARGET_SIZE                   = 768
NUM_INFERENCE_STEPS           = 40
GUIDANCE_SCALE                = 12.0
CONTROLNET_CONDITIONING_SCALE = 1.3

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Validate
assert Path(INPUT_IMAGE_PATH).exists(), (
    f"File not found: {INPUT_IMAGE_PATH}\n"
    "Upload your CAD image via the Files panel on the left sidebar."
)

import matplotlib.pyplot as plt
from PIL import Image
input_image = Image.open(INPUT_IMAGE_PATH).convert("RGB")
print(f"Image loaded: {input_image.size}")
plt.figure(figsize=(6,5)); plt.imshow(input_image); plt.axis("off"); plt.title("Input"); plt.show()

## Cell 4 — Download / Load Models from Drive
First run downloads ~6 GB. Every run after loads from Drive in ~30s.

In [ ]:
import time
from huggingface_hub import snapshot_download

REPOS = {
    "controlnet" : "lllyasviel/control_v11p_sd15_lineart",
    "sd15"       : "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "annotators" : "lllyasviel/Annotators",
}

def ensure_cached(name, repo, local_path):
    local_path = Path(local_path)
    marker = local_path / ".download_complete"
    if marker.exists():
        print(f"  [{name}] Loaded from Drive cache")
        return str(local_path)
    print(f"  [{name}] Downloading {repo} → Drive (first time only)...")
    local_path.mkdir(parents=True, exist_ok=True)
    try:
        snapshot_download(
            repo_id=repo,
            local_dir=str(local_path),
            ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*", "rust_model*"],
        )
        marker.touch()
        print(f"  [{name}] Done — cached to Drive")
    except Exception as e:
        print(f"  [{name}] Download failed ({e}), will use HF hub directly")
        return repo
    return str(local_path)

t0 = time.time()
resolved = {name: ensure_cached(name, repo, PATHS[name]) for name, repo in REPOS.items()}
print(f"\nAll models ready in {time.time()-t0:.0f}s")

## Cell 5 — Build Control Map

In [ ]:
from controlnet_aux import LineartDetector
import numpy as np

def resize_for_sd(img, size):
    w, h = img.size
    scale = size / max(w, h)
    return img.resize((int(round(w*scale/8))*8, int(round(h*scale/8))*8), Image.LANCZOS)

print("Building line-art control map...")
try:
    detector = LineartDetector.from_pretrained(resolved["annotators"])
except Exception:
    detector = LineartDetector.from_pretrained("lllyasviel/Annotators")

input_resized = resize_for_sd(input_image, TARGET_SIZE)
control_map   = detector(input_resized, coarse=False)
control_map.save(OUTPUT_DIR / "control_map.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(input_resized); axes[0].set_title("Resized Input"); axes[0].axis("off")
axes[1].imshow(control_map);   axes[1].set_title("Control Map");   axes[1].axis("off")
plt.tight_layout(); plt.show()
print(f"Control map: {control_map.size}")

## Cell 6 — Load Pipeline

In [ ]:
import gc, torch, warnings
warnings.filterwarnings("ignore")
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

device      = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

if device == "cpu":
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  Free VRAM: {free_gb:.1f} GB")

print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    resolved["controlnet"], torch_dtype=torch_dtype
)

print("Loading SD 1.5...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    resolved["sd15"],
    controlnet=controlnet,
    torch_dtype=torch_dtype,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

# Best available memory optimisation
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xformers: enabled")
except Exception:
    pipe.enable_attention_slicing(1)
    print("xformers unavailable — attention slicing enabled")

free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Pipeline ready  |  Free VRAM: {free_gb:.1f} GB")

## Cell 7 — Generate

In [ ]:
generator = torch.Generator(device=device).manual_seed(SEED)

print(f"Steps: {NUM_INFERENCE_STEPS}  |  Images: {NUM_IMAGES}  |  Size: {control_map.size}")
print("Generating (~2-3 min)...")

t0 = time.time()
try:
    result = pipe(
        prompt                        = PROMPT,
        negative_prompt               = NEGATIVE_PROMPT,
        image                         = control_map,
        num_inference_steps           = NUM_INFERENCE_STEPS,
        guidance_scale                = GUIDANCE_SCALE,
        controlnet_conditioning_scale = CONTROLNET_CONDITIONING_SCALE,
        num_images_per_prompt         = NUM_IMAGES,
        generator                     = generator,
    )
    images = result.images

except torch.cuda.OutOfMemoryError:
    print("OOM — retrying one image at a time with CPU offload...")
    gc.collect(); torch.cuda.empty_cache()
    pipe.enable_sequential_cpu_offload()
    generator = torch.Generator(device="cpu").manual_seed(SEED)
    result = pipe(
        prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT,
        image=control_map, num_inference_steps=25,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=CONTROLNET_CONDITIONING_SCALE,
        num_images_per_prompt=1, generator=generator,
    )
    images = result.images

print(f"Done in {time.time()-t0:.0f}s")

## Cell 8 — Display & Save

In [ ]:
from datetime import datetime

cols = min(len(images), 4)
rows = (len(images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5))
axes = np.array(axes).flatten()

run_id      = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_out   = CACHE_ROOT / "renders"
drive_out.mkdir(exist_ok=True)

for i, img in enumerate(images):
    axes[i].imshow(img); axes[i].set_title(f"Render {i+1}"); axes[i].axis("off")
    img.save(OUTPUT_DIR  / f"render_{i+1:02d}.png")
    img.save(drive_out   / f"{run_id}_render_{i+1:02d}.png")

for ax in axes[len(images):]: ax.set_visible(False)
plt.tight_layout(); plt.show()
print(f"Saved to /content/outputs/ and Drive/colab_model_cache/renders/{run_id}_render_*.png")

---
## Quick Reference

| Parameter | What it does | Range |
|---|---|---|
| `CONTROLNET_CONDITIONING_SCALE` | Shape accuracy — higher = closer to CAD lines | 1.0 – 1.5 |
| `GUIDANCE_SCALE` | Prompt strictness | 10 – 14 |
| `SEED` | Change for different variations | any int |
| `NUM_INFERENCE_STEPS` | Quality vs speed | 30 – 50 |

**Drive storage used:** ~6 GB total
```
colab_model_cache/
  ├─ controlnet-lineart/    ~1.5 GB
  ├─ stable-diffusion-v1-5/ ~4.0 GB
  ├─ annotators/            ~0.3 GB
  └─ renders/               your outputs
```